In [21]:
!pip install pyspark -q
!pip install boto3 -q
!pip install pyarrow -q
!pip install databricks-sql-connector pandas -q

In [22]:
from google.colab import drive
import os

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
    print("✅ Drive mounted successfully")
else:
    print("✅ Drive already mounted, skipping...")

✅ Drive already mounted, skipping...


In [23]:
#configuration notebook

import os
os.chdir("/content/drive/My Drive/Colab Notebooks")
%run oo_config.ipynb

In [24]:
import boto3
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from functools import reduce
from pyspark.sql import DataFrame

# ── Config ────────────────────────────────────────────────────────────────────
MINIO_BUCKET      = "rawdatasets"
MINIO_CSV_PREFIX  = "dividend/"
MINIO_CKPT_PREFIX = "dividend-spark/checkpoint/"

LOCAL_CSV         = "/tmp/dividend/csv_input"
LOCAL_CHECKPOINT  = "/tmp/dividend/checkpoint"
OUTPUT            = "/content/drive/MyDrive/dividend/spark/output"

os.makedirs(LOCAL_CSV, exist_ok=True)
os.makedirs(LOCAL_CHECKPOINT, exist_ok=True)
os.makedirs(OUTPUT, exist_ok=True)


# ── boto3 Client ──────────────────────────────────────────────────────────────
s3 = boto3.client(
    "s3",
    endpoint_url=G_MINIO_ENDPOINT,
    aws_access_key_id=G_MINIO_ACCESS_KEY,
    aws_secret_access_key=G_MINIO_SECRET_KEY
)

# ── Helper: MinIO ↔ Local Sync ────────────────────────────────────────────────
def download_folder_from_minio(bucket, prefix, local_dir):
    response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
    for obj in response.get("Contents", []):
        s3_key = obj["Key"]
        if s3_key.endswith("/"):
            continue
        relative   = os.path.relpath(s3_key, prefix)
        local_path = os.path.join(local_dir, relative)
        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        s3.download_file(Bucket=bucket, Key=s3_key, Filename=local_path)
    print(f"✅ Checkpoint restored from MinIO [{prefix}] → {local_dir}")

def upload_folder_to_minio(bucket, prefix, local_dir):
    for root, _, files in os.walk(local_dir):
        for file in files:
            local_path = os.path.join(root, file)
            relative   = os.path.relpath(local_path, local_dir)
            s3_key     = os.path.join(prefix, relative).replace("\\", "/")
            s3.upload_file(Filename=local_path, Bucket=bucket, Key=s3_key)
    print(f"✅ Checkpoint saved to MinIO [{prefix}] ← {local_dir}")

# ── Step 1: Pull CSVs from MinIO → Local /tmp ────────────────────────────────
response = s3.list_objects_v2(Bucket=MINIO_BUCKET, Prefix=MINIO_CSV_PREFIX)
for obj in response.get("Contents", []):
    s3_key = obj["Key"]
    if s3_key.endswith("/"):
        continue
    filename   = os.path.basename(s3_key)
    local_path = os.path.join(LOCAL_CSV, filename)
    if os.path.exists(local_path):
        continue
    s3.download_file(Bucket=MINIO_BUCKET, Key=s3_key, Filename=local_path)
    print(f"Downloaded: {s3_key} → {local_path}")

# ── Step 2: Restore Checkpoint from MinIO → Local /tmp ───────────────────────
download_folder_from_minio(MINIO_BUCKET, MINIO_CKPT_PREFIX, LOCAL_CHECKPOINT)

# ── Step 3: Spark Session ─────────────────────────────────────────────────────
spark = SparkSession.builder \
    .appName("DividendStreamIncremental") \
    .master("local[*]") \
    .getOrCreate()

# ── Schema ────────────────────────────────────────────────────────────────────
schema = StructType([
    StructField("date",                StringType(), True),
    StructField("transaction_details", StringType(), True),
    StructField("currency",            StringType(), True),
    StructField("money_in",            StringType(), True)
])

# ── Step 4: readStream from Local CSV ────────────────────────────────────────
df_stream = spark.readStream \
    .option("header", "true") \
    .schema(schema) \
    .csv(LOCAL_CSV)

df_stream = df_stream \
    .withColumn("source_file", F.element_at(F.split(F.input_file_name(), "/"), -1)) \
    .withColumn("date_modified", F.current_timestamp())

# ── Step 5: Capture New Rows via foreachBatch ─────────────────────────────────
incremental_batches = []

def capture_batch(batch_df, batch_id):
    # Persist to Drive for full history
    batch_df.write \
        .mode("append") \
        .parquet(OUTPUT)
    # Keep reference to new rows only
    incremental_batches.append(batch_df)
    print(f"  Batch {batch_id}: {batch_df.count()} new rows")

query = df_stream.writeStream \
    .trigger(availableNow=True) \
    .foreachBatch(capture_batch) \
    .option("checkpointLocation", LOCAL_CHECKPOINT) \
    .start()

query.awaitTermination()
print("✅ Incremental load complete.")

# ── Step 6: Push Updated Checkpoint back to MinIO ────────────────────────────
upload_folder_to_minio(MINIO_BUCKET, MINIO_CKPT_PREFIX, LOCAL_CHECKPOINT)

# ── Step 7: final_df = only new rows from this run ───────────────────────────
final_df = reduce(DataFrame.union, incremental_batches) if incremental_batches else spark.createDataFrame([], schema)
print(f"New rows this run: {final_df.count()}")
final_df.show()
final_df.count()

# ── Step 8: full_df = all accumulated history (optional) ─────────────────────
# full_df = spark.read.parquet(OUTPUT)
# print(f"Total rows all time: {full_df.count()}")
# full_df.show()


✅ Checkpoint restored from MinIO [dividend-spark/checkpoint/] → /tmp/dividend/checkpoint
✅ Incremental load complete.
✅ Checkpoint saved to MinIO [dividend-spark/checkpoint/] ← /tmp/dividend/checkpoint
New rows this run: 0
+----+-------------------+--------+--------+
|date|transaction_details|currency|money_in|
+----+-------------------+--------+--------+
+----+-------------------+--------+--------+



0

In [25]:
class StopExecution(Exception):
    def _render_traceback_(self):
        return []  # suppresses the red traceback entirely

if final_df.count() <= 0 :
  print("🛑 Pipeline stopped: No data to proceed")
  raise StopExecution()


🛑 Pipeline stopped: No data to proceed


StopExecution: 

In [ ]:
# dataframe changes
from pyspark.sql.functions import udf
from pyspark.sql.types import DateType
from dateutil import parser

final_df = final_df.toDF("date", "transaction_details", "currency", "money_in","source_file","date_modified")
final_df = final_df.dropna(how='any', thresh=None, subset=["date"])

#df.printSchema()
#df.show(100, truncate=False)

In [ ]:
#Check date valid
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType
from dateutil import parser

def try_parse_date(date_str):
    if date_str is None:
        return "NULL"
    try:
        parsed = parser.parse(date_str.strip())
        return parsed.strftime("%Y-%m-%d")  # return standardized date if valid
    except:
        return f"INVALID: {date_str}"  # return original value flagged as invalid

audit_udf = udf(try_parse_date, StringType())

df_date = final_df.withColumn("date_audit", audit_udf("date"))
#df_date.show(truncate=False)

valid_df   = df_date.filter(~col("date_audit").startswith("INVALID") & (col("date_audit") != "NULL"))
invalid_df = df_date.filter((col("date_audit").startswith("INVALID") | (col("date_audit") == "NULL")) & (col("date_audit") != "INVALID: Date"))

print("=== VALID DATES ===")
# valid_df.select("date", "date_audit").show(truncate=False)
valid_df = valid_df.withColumn("money_in", col("money_in").cast("double"))
valid_df = valid_df.withColumn("date_audit", col("date_audit").cast("date"))
valid_df.show(truncate=False)
valid_df.printSchema()
valid_df.count()

print("=== INVALID DATES ===")
invalid_df.select("date", "date_audit").show(truncate=False)

In [ ]:
# ─── Databricks Connection Config ───────────────────────────────────────────
SERVER_HOSTNAME = G_SERVER_HOSTNAME
HTTP_PATH       = G_HTTP_PATH   # from Connection Details tab
ACCESS_TOKEN    = G_ACCESS_TOKEN         # replace with your PAT

# ─── Catalog / Schema / Table Target ────────────────────────────────────────
CATALOG   = "workspace"            # change to your catalog name (Unity Catalog)
SCHEMA    = "default"         # change to your schema/database
TABLE     = "dividend"
FULL_TABLE = f"{CATALOG}.{SCHEMA}.{TABLE}"

In [ ]:
from databricks import sql

df_to_insert = valid_df

def run_query(cursor, query):
    """Helper to execute and optionally fetch results."""
    cursor.execute(query)
    try:
        return cursor.fetchall()
    except Exception:
        return None

COLUMNS = [
    "date",
    "transaction_details",
    "currency",
    "money_in",
    "source_file",
    "date_modified",
    "date_audit"
]

placeholders = ", ".join(["?"] * len(COLUMNS))
col_names    = ", ".join(COLUMNS)

with sql.connect(
    server_hostname = SERVER_HOSTNAME,
    http_path       = HTTP_PATH,
    access_token    = ACCESS_TOKEN
) as conn:
    with conn.cursor() as cursor:

        # ── Step 1: Set catalog and schema context ───────────────────────────
        cursor.execute(f"USE CATALOG {CATALOG}")
        cursor.execute(f"USE SCHEMA {SCHEMA}")

        # ── Step 2: Create table if not exists ───────────────────────────────
        create_sql = f"""
        CREATE TABLE IF NOT EXISTS {FULL_TABLE} (
                date                STRING,
                transaction_details STRING,
                currency            STRING,
                money_in            DOUBLE,
                source_file         STRING,
                date_modified       DATE,
                date_audit          DATE
        )
        """
        cursor.execute(create_sql)
        print(f"✅ Table '{FULL_TABLE}' created/verified.")

        # ── Step 3: Insert rows from Pandas DataFrame ─────────────────────────

        # Spark equivalent of: [tuple(r) for r in df.itertuples(index=False)]
        rows = [tuple(row) for row in df_to_insert.collect()]

        insert_sql = f"""
        INSERT INTO {FULL_TABLE} ({col_names})
        SELECT {placeholders}
        WHERE NOT EXISTS (
            SELECT 1 FROM {FULL_TABLE}
            WHERE date = ?
              AND transaction_details = ?
        )
        """


        # Append the 3 key values at end of each row for the WHERE NOT EXISTS check
        rows_with_keys = [
            row + (row[COLUMNS.index("date")],
                  row[COLUMNS.index("transaction_details")]
            )
            for row in rows
        ]

        cursor.executemany(insert_sql, rows_with_keys)

        print(f"✅ Inserted {df_to_insert.count()} rows into '{FULL_TABLE}'.")

        # ── Step 4: Verify by reading back ───────────────────────────────────
        results = run_query(cursor, f"SELECT * FROM {FULL_TABLE}")
        print(f"\n📦 Data in '{FULL_TABLE}':")
        for r in results:
            print(r)

In [ ]:
spark.stop()

In [ ]:
%%script echo "Skipping..."
#reset to start from scratch
import boto3
import os
import shutil

# ── boto3 Client ──────────────────────────────────────────────────────────────
s3 = boto3.client(
    "s3",
    endpoint_url=G_MINIO_ENDPOINT,
    aws_access_key_id=G_MINIO_ACCESS_KEY,
    aws_secret_access_key=G_MINIO_SECRET_KEY
)

MINIO_BUCKET      = "rawdatasets"
MINIO_CKPT_PREFIX = "dividend-spark/checkpoint/"
LOCAL_CSV         = "/tmp/dividend/csv_input"
LOCAL_CHECKPOINT  = "/tmp/dividend/checkpoint"
OUTPUT            = "/content/drive/MyDrive/dividend/spark/output"

# 1. Delete checkpoint from MinIO
response = s3.list_objects_v2(Bucket=MINIO_BUCKET, Prefix=MINIO_CKPT_PREFIX)
objects  = [{"Key": obj["Key"]} for obj in response.get("Contents", [])]
if objects:
    s3.delete_objects(Bucket=MINIO_BUCKET, Delete={"Objects": objects})
    print(f"✅ Cleared MinIO checkpoint: {MINIO_CKPT_PREFIX}")
else:
    print("ℹ️  No MinIO checkpoint found.")

# 2. Delete local /tmp cache
if os.path.exists(LOCAL_CSV):
    shutil.rmtree(LOCAL_CSV)
    print(f"✅ Cleared local CSV cache: {LOCAL_CSV}")

if os.path.exists(LOCAL_CHECKPOINT):
    shutil.rmtree(LOCAL_CHECKPOINT)
    print(f"✅ Cleared local checkpoint: {LOCAL_CHECKPOINT}")

# 3. Delete Drive output (Parquet history)
if os.path.exists(OUTPUT):
    shutil.rmtree(OUTPUT)
    print(f"✅ Cleared Drive output: {OUTPUT}")